## Final Dataset and Sanity check for Severity Analysis

In [2]:
import pandas as pd
import sqlite3

In [3]:
conn = sqlite3.connect("vic_crash_data.db")

In [4]:
final_data = pd.read_sql("""
SELECT 
    -- Accidents core
    a.accident_no,
    a.accident_type,
    a.day_of_week,
    a.dca_code,
    a.no_of_vehicles,
    a.police_attended,
    a.road_geometry,
    a.light_condition,
    a.speed_zone,
    a.severity,
    
    -- Date/time (extract in pandas)
    a.accident_date,
    a.accident_time,

    -- Node info
    n.node_type,
    n.deg_urban_name,
    n.lga_name,

    -- Road info
    an.road_type,

    -- Vehicle aggregates
    COUNT(DISTINCT v.vehicle_id)   as total_vehicles,
    MAX(v.level_of_damage)         as max_vehicle_damage,

    -- Person aggregates
    COUNT(DISTINCT p.person_id)    as total_persons

FROM accidents a
LEFT JOIN accident_node an  ON a.accident_no = an.accident_no
LEFT JOIN road_node n       ON an.node_id    = n.node_id
LEFT JOIN vehicle_info v    ON a.accident_no = v.accident_no
LEFT JOIN person_info p     ON a.accident_no = p.accident_no

GROUP BY a.accident_no
""", conn)

final_data.head()

,accident_no,accident_type,day_of_week,dca_code,no_of_vehicles,police_attended,road_geometry,light_condition,speed_zone,severity,accident_date,accident_time,node_type,deg_urban_name,lga_name,road_type,total_vehicles,max_vehicle_damage,total_persons
0,T20120000009,4,1,171,1,1,5,5,100,3,2012-01-01,02:25:00.000000,N,RURAL_VICTORIA,BAW BAW,ROAD,1,5,2
1,T20120000012,1,1,110,2,1,1,3,80,2,2012-01-01,02:00:00.000000,N,MELB_URBAN,MONASH,ROAD,2,4,3
2,T20120000013,1,1,160,2,1,2,3,60,2,2012-01-01,03:35:00.000000,I,MELB_URBAN,KINGSTON,ROAD,2,4,1
3,T20120000018,4,1,173,1,1,1,5,100,3,2012-01-01,05:15:00.000000,I,RURAL_VICTORIA,MILDURA,HIGHWAY,1,5,1
4,T20120000021,4,1,171,1,1,5,1,50,3,2012-01-01,07:30:00.000000,N,MELB_URBAN,MORNINGTON PENINSULA,ROAD,1,3,3


In [5]:
final_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 191779 entries, 0 to 191778
Data columns (total 19 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   accident_no         191779 non-null  object
 1   accident_type       191779 non-null  object
 2   day_of_week         191779 non-null  object
 3   dca_code            191779 non-null  object
 4   no_of_vehicles      191779 non-null  int64 
 5   police_attended     191779 non-null  int64 
 6   road_geometry       191779 non-null  object
 7   light_condition     191779 non-null  object
 8   speed_zone          191779 non-null  int64 
 9   severity            191779 non-null  object
 10  accident_date       191779 non-null  object
 11  accident_time       191779 non-null  object
 12  node_type           191689 non-null  object
 13  deg_urban_name      190791 non-null  object
 14  lga_name            191689 non-null  object
 15  road_type           189191 non-null  object
 16  to

### Length matches from number of accidents explored in EDA02

In [6]:
len(final_data)

191779

In [7]:
final_data.isna().sum().sort_values()

accident_no              0
total_vehicles           0
accident_time            0
accident_date            0
speed_zone               0
light_condition          0
severity                 0
police_attended          0
no_of_vehicles           0
dca_code                 0
day_of_week              0
accident_type            0
road_geometry            0
total_persons            0
max_vehicle_damage       9
node_type               90
lga_name                90
deg_urban_name         988
road_type             2588
dtype: int64

In [8]:
final_data["accident_no"].duplicated().sum()

0

In [9]:
final_data["severity"].value_counts()

severity
3    121239
2     67329
1      3207
4         4
Name: count, dtype: int64

In [10]:
final_data["road_type"].value_counts()

road_type
ROAD       96469
STREET     42291
HIGHWAY    17158
FREEWAY     8020
DRIVE       5271
           ...  
CUTTING        1
PASS           1
VALLEY         1
ROUTE          1
NOOK           1
Name: count, Length: 80, dtype: int64

In [11]:
final_data["road_type"].value_counts()[final_data["road_type"].value_counts() > 1000]

road_type
ROAD         96469
STREET       42291
HIGHWAY      17158
FREEWAY       8020
DRIVE         5271
AVENUE        5230
RAMP          2732
PARADE        2269
BOULEVARD     1742
TRACK         1220
WAY           1196
CRESCENT      1116
LANE          1082
Name: count, dtype: int64

In [12]:
final_data["dca_code"].value_counts()

dca_code
130    34715
121    16480
110    13716
171    10817
113     8915
       ...  
118       26
193       25
161       12
117       10
125        4
Name: count, Length: 81, dtype: int64

In [13]:
final_data["lga_name"].value_counts()

lga_name
MELBOURNE           10005
CASEY                8654
GEELONG              7505
HUME                 6912
DANDENONG            6607
                    ...  
(LAKE MOUNTAIN)        24
(MOUNT BULLER)         23
(MOUNT BAW BAW)        13
(FRENCH ISLAND)         4
(MOUNT STIRLING)        2
Name: count, Length: 87, dtype: int64

In [14]:
for col in ['lga_name', 'dca_code', 'road_type']:
    print(f"{col}: {final_data[col].nunique()} unique values")
    print(final_data[col].value_counts().head(10))
    print()

lga_name: 87 unique values
lga_name
MELBOURNE       10005
CASEY            8654
GEELONG          7505
HUME             6912
DANDENONG        6607
BRIMBANK         5925
MONASH           5743
WHITTLESEA       5681
MORELAND         5264
YARRA RANGES     5228
Name: count, dtype: int64

dca_code: 81 unique values
dca_code
130    34715
121    16480
110    13716
171    10817
113     8915
174     7501
173     7487
120     6135
100     6065
181     5051
Name: count, dtype: int64

road_type: 80 unique values
road_type
ROAD         96469
STREET       42291
HIGHWAY      17158
FREEWAY       8020
DRIVE         5271
AVENUE        5230
RAMP          2732
PARADE        2269
BOULEVARD     1742
TRACK         1220
Name: count, dtype: int64



In [15]:
for col in ['lga_name', 'dca_code', 'road_type']:
    top10_coverage = final_data[col].value_counts().nlargest(10).sum() / len(final_data) * 100
    print(f"{col} top 10 coverage: {top10_coverage:.1f}%")

lga_name top 10 coverage: 35.2%
dca_code top 10 coverage: 60.9%
road_type top 10 coverage: 95.1%


In [16]:
final_data['deg_urban_name'].value_counts()

deg_urban_name
MELB_URBAN                 119553
RURAL_VICTORIA              41090
LARGE_PROVINCIAL_CITIES     10895
SMALL_CITIES                 9666
TOWNS                        5788
MELBOURNE_CBD                2020
SMALL_TOWNS                  1779
Name: count, dtype: int64

In [17]:
top20_coverage = final_data['dca_code'].value_counts().nlargest(25).sum() / len(final_data) * 100
print(f"dca_code top 20 coverage: {top20_coverage:.1f}%")

dca_code top 20 coverage: 83.6%


In [18]:
final_data['severity'].value_counts()
final_data['severity'].value_counts(normalize=True) * 100  # percentages

severity
3    63.218079
2    35.107598
1     1.672237
4     0.002086
Name: proportion, dtype: float64

In [19]:
# Are the 2588 road_type nulls random or concentrated in certain severity classes?
final_data[final_data['road_type'].isna()]['severity'].value_counts()

# Same for deg_urban_name
final_data[final_data['deg_urban_name'].isna()]['severity'].value_counts()

severity
3    522
2    466
Name: count, dtype: int64

In [20]:
final_data["speed_zone"].value_counts()

speed_zone
60     62693
50     31622
80     28976
100    27016
999    12571
40     12304
70     12092
110     1976
888     1258
90       498
30       389
777      364
75        20
Name: count, dtype: int64

In [21]:
final_data["max_vehicle_damage"].value_counts()

max_vehicle_damage
4    39661
3    34370
5    32180
1    24536
6    23403
9    21480
2    16140
Name: count, dtype: int64

In [22]:
final_data["accident_time"].head()

0    02:25:00.000000
1    02:00:00.000000
2    03:35:00.000000
3    05:15:00.000000
4    07:30:00.000000
Name: accident_time, dtype: object

In [23]:
for col in ['accident_type', 'day_of_week', 'road_geometry', 
            'light_condition', 'police_attended', 'node_type']:
    print(f"\n{col}: {final_data[col].nunique()} unique values")
    print(final_data[col].value_counts())


accident_type: 9 unique values
accident_type
1    124694
4     28797
2     15790
6      9017
8      7925
3      2095
5      2012
7      1240
9       209
Name: count, dtype: int64

day_of_week: 8 unique values
day_of_week
5    29434
6    29052
4    28611
3    27967
2    26623
1    24119
7    21200
0     4773
Name: count, dtype: int64

road_geometry: 9 unique values
road_geometry
5    99419
1    44128
2    43437
4     3667
3      663
9      273
6      175
8       11
7        6
Name: count, dtype: int64

light_condition: 7 unique values
light_condition
1    128654
3     29248
2     16084
5     10064
9      5134
6      2184
4       411
Name: count, dtype: int64

police_attended: 3 unique values
police_attended
1    141312
2     49779
9       688
Name: count, dtype: int64

node_type: 3 unique values
node_type
N    111073
I     80159
O       457
Name: count, dtype: int64


In [24]:
final_data.dtypes

accident_no           object
accident_type         object
day_of_week           object
dca_code              object
no_of_vehicles         int64
police_attended        int64
road_geometry         object
light_condition       object
speed_zone             int64
severity              object
accident_date         object
accident_time         object
node_type             object
deg_urban_name        object
lga_name              object
road_type             object
total_vehicles         int64
max_vehicle_damage    object
total_persons          int64
dtype: object